# FloydNet: Routed k-Floyd training and inference

Notebook рассчитан на запуск из корня репозитория `FloydNet`. Реализуется learned pivot routing: для каждого k-tuple выбираются `top_k` pivot-реляций, после чего pivotal attention считается только на выбранных pivot-ах.


In [ ]:
import os, sys

IN_COLAB = 'google.colab' in sys.modules

REPO_URL = 'https://github.com/artlvruran/FloydNet.git'
REPO_DIR = 'FloydNet'
BRANCH = 'main'

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        !git clone --branch $BRANCH --single-branch $REPO_URL $REPO_DIR
    else:
        !cd $REPO_DIR && git fetch origin $BRANCH && git checkout $BRANCH && git pull origin $BRANCH
    !pip install -q jax optax networkx matplotlib pandas pulp
    sys.path.insert(0, REPO_DIR)
else:
    # Локальный запуск: предполагается, что ноутбук уже лежит внутри
    # чекаута репозитория (ветка gnarl).
    sys.path.insert(0, '.')

print('Repo ready, sys.path[0] =', sys.path[0])


In [ ]:
from pathlib import Path
import sys
import os

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src' / 'floydnet').exists() and (REPO_ROOT / 'FloydNet' / 'src' / 'floydnet').exists():
    REPO_ROOT = REPO_ROOT / 'FloydNet'
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'example'))
os.chdir(REPO_ROOT)
print(REPO_ROOT)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.loader import DataLoader
from torch.nn import CosineEmbeddingLoss
from BREC.BRECDataset_v3 import BRECDataset
from floydnet import RoutedPivotalAttentionBlock

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_default_dtype(torch.float64)
torch_geometric.seed_everything(2023)
print(device)


In [ ]:
FLOYD_LEVEL = 2
N_EMBD = 64
N_HEAD = 8
DEPTH = 8
TOP_K = 8
ROUTE_DIM = 16
PIVOT_CHUNK_SIZE = 32
DROPOUT = 0.0
ROUTER_WEIGHT = 1.0
ROUTER_TEMPERATURE = 1.0
LR = 1e-3
WEIGHT_DECAY = 0.0
EPOCHS = 20
BATCH_SIZE = 8
GRAPH_ID = 0
SAMPLE_NUM = 400
NUM_RELABEL = 32
CHECKPOINT = REPO_ROOT / 'outputs' / 'routed' / f'breс_id_{GRAPH_ID}_k{FLOYD_LEVEL}.pt'
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
print({k: v for k, v in [('FLOYD_LEVEL', FLOYD_LEVEL), ('TOP_K', TOP_K), ('N_EMBD', N_EMBD), ('N_HEAD', N_HEAD)]})


In [32]:
import torch
from torch import nn
from floydnet import RoutedPivotalAttentionBlock
from common.graph import graph_preprocess


class RoutedFloydNet(nn.Module):
    def __init__(self, n_embd, n_head, depth, floyd_level, vocab_size=128):
        super().__init__()

        self.n_embd = n_embd
        self.floyd_level = floyd_level

        self.emb = nn.Embedding(vocab_size, n_embd)

        self.norms = nn.ModuleList([
            nn.LayerNorm(n_embd)
            for _ in range(depth)
        ])

        self.blocks = nn.ModuleList([
            RoutedPivotalAttentionBlock(
                embed_dim=n_embd,
                num_heads=n_head,
                floyd_level=floyd_level,
                top_k=TOP_K,
                route_dim=ROUTE_DIM,
                pivot_chunk_size=PIVOT_CHUNK_SIZE,
                dropout=DROPOUT,
                router_weight=ROUTER_WEIGHT,
                router_temperature=ROUTER_TEMPERATURE,
            )
            for _ in range(depth)
        ])

        self.head = nn.Linear(n_embd, n_embd)

        for p in self.emb.parameters():
            p.requires_grad = False

        for block in self.blocks:
            block.c_qkv.weight.requires_grad = False
            block.c_proj.weight.requires_grad = False

            if block.c_qkv.bias is not None:
                block.c_qkv.bias.requires_grad = False
            if block.c_proj.bias is not None:
                block.c_proj.bias.requires_grad = False

            block.route_q.weight.requires_grad = True
            block.route_k.weight.requires_grad = True

        for p in self.norms.parameters():
            p.requires_grad = True

        for p in self.head.parameters():
            p.requires_grad = True

    def create_hypergraph(self, x):
        x_new = torch.zeros(
            [x.shape[0], x.shape[1], x.shape[1], x.shape[1]],
            device=x.device,
            dtype=x.dtype,
        )

        x_new = x_new * 2 + x[:, :, :, None]
        x_new = x_new * 2 + x[:, :, None, :]
        x_new = x_new * 2 + x[:, None, :, :]

        xt = x.transpose(-2, -1)

        x_new = x_new * 2 + xt[:, :, :, None]
        x_new = x_new * 2 + xt[:, :, None, :]
        x_new = x_new * 2 + xt[:, None, :, :]

        return x_new

    def relation_tensor(self, graph):
        x = graph.adj.long()

        if self.floyd_level == 2:
            return x

        if self.floyd_level == 3:
            return self.create_hypergraph(x)

        raise ValueError("BREC model supports floyd_level 2 or 3")

    def forward(self, graph, return_routing=False):
        graph = graph_preprocess(graph)

        x = self.relation_tensor(graph)
        x = self.emb(x)

        routing = []

        for norm, block in zip(self.norms, self.blocks):
            if return_routing:
                y, idx, scores = block(
                    norm(x),
                    return_routing=True
                )
                x = x + y
                routing.append((idx, scores))
            else:
                x = x + block(norm(x))

        dims = tuple(range(1, x.dim() - 1))
        x = x.mean(dim=dims)
        x = self.head(x)

        if return_routing:
            return x, routing

        return x

In [33]:
model = RoutedFloydNet(N_EMBD, N_HEAD, DEPTH, FLOYD_LEVEL).to(device).double()
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('parameters:', n_total)
print('trainable:', n_train)


parameters: 212032
trainable: 7232


In [22]:
%cd /content/FloydNet/example/data/BREC

!mkdir -p raw
%cd raw

!wget -O BREC_data_all.zip https://raw.githubusercontent.com/GraphPKU/BREC/Release/BREC_data_all.zip
!unzip -o BREC_data_all.zip

/content/FloydNet/example/data/BREC
/content/FloydNet/example/data/BREC/raw
--2026-10-01 11:58:30--  https://raw.githubusercontent.com/GraphPKU/BREC/Release/BREC_data_all.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7676292 (7.3M) [application/zip]
Saving to: ‘BREC_data_all.zip’

BREC_data_all.zip   100%[===================>]   7.32M  --.-KB/s    in 0.05s   

2026-10-01 11:58:30 (134 MB/s) - ‘BREC_data_all.zip’ saved [7676292/7676292]

Archive:  BREC_data_all.zip
  inflating: brec_v3.npy             
  inflating: brec_v3_3wl.npy         
  inflating: brec_v3_no4v_60cfi.npy  


In [23]:
%cd /content/FloydNet/example

/content/FloydNet/example


In [24]:
from BREC.BRECDataset_v3 import BRECDataset
import torch

dataset = BRECDataset(
    root="data/BREC",
    transform=lambda g: setattr(g, "x", torch.ones(g.num_nodes, 1)) or g
)

print("dataset size:", len(dataset))

Processing...


dataset size: 51200


Done!


In [25]:

dataset_traintest = dataset[GRAPH_ID * NUM_RELABEL * 2:(GRAPH_ID + 1) * NUM_RELABEL * 2]
dataset_reliability = dataset[(GRAPH_ID + SAMPLE_NUM) * NUM_RELABEL * 2:(GRAPH_ID + SAMPLE_NUM + 1) * NUM_RELABEL * 2]
print('train/test samples:', len(dataset_traintest))
print('reliability samples:', len(dataset_reliability))


train/test samples: 64
reliability samples: 64


In [26]:
%cd /content/

/content


## Training

Для выбранного `GRAPH_ID` используются 32 relabel-пары. В отличие от исходного BREC-кода здесь обучаются и основной routed block, и router.

In [34]:
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.98),
)

loss_fn = CosineEmbeddingLoss(margin=0.0)
loader = DataLoader(dataset_traintest, batch_size=BATCH_SIZE, shuffle=True)

for p in model.emb.parameters():
    p.requires_grad = False

for p in model.blocks.parameters():
    p.requires_grad = False

for p in model.head.parameters():
    p.requires_grad = True

history = []

model.train()

for epoch in range(EPOCHS):
    total = 0.0
    count = 0

    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()

        pred = model(data)

        y = torch.full(
            (pred.shape[0] // 2,),
            -1.0,
            device=device,
            dtype=pred.dtype,
        )

        loss = loss_fn(pred[0::2], pred[1::2], y)
        loss.backward()
        optimizer.step()

        total += loss.item() * (pred.shape[0] // 2)
        count += pred.shape[0] // 2

    epoch_loss = total / count
    history.append(epoch_loss)

    print(f"epoch {epoch + 1:03d}: loss={epoch_loss:.6f}")

epoch 001: loss=0.999712
epoch 002: loss=0.998227
epoch 003: loss=0.966489
epoch 004: loss=0.734610
epoch 005: loss=0.724562
epoch 006: loss=0.755986
epoch 007: loss=0.682621
epoch 008: loss=0.660632
epoch 009: loss=0.490057
epoch 010: loss=0.800688
epoch 011: loss=0.805498
epoch 012: loss=0.593140
epoch 013: loss=0.562390
epoch 014: loss=0.942240
epoch 015: loss=0.950810
epoch 016: loss=0.911424
epoch 017: loss=0.600337
epoch 018: loss=0.610732
epoch 019: loss=0.544654
epoch 020: loss=0.791141


In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(range(1, len(history) + 1), history)
plt.xlabel('epoch')
plt.ylabel('CosineEmbeddingLoss')
plt.tight_layout()
plt.show()


## Inference

Можно либо использовать объект `model` после обучения, либо загрузить checkpoint в чистом kernel.

In [ ]:
ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=False)
model = RoutedFloydNet(
    ckpt['config']['n_embd'],
    ckpt['config']['n_head'],
    ckpt['config']['depth'],
    ckpt['config']['floyd_level'],
).to(device).double()
model.load_state_dict(ckpt['model'])
model.eval()

@torch.no_grad()
def infer(dataset_part, batch_size=1, return_routing=False):
    loader = DataLoader(dataset_part, batch_size=batch_size, shuffle=False)
    preds = []
    routes = []
    for data in loader:
        data = data.to(device)
        if return_routing:
            pred, routing = model(data, return_routing=True)
            routes.append(routing)
        else:
            pred = model(data)
        preds.append(pred)
    pred = torch.cat(preds, dim=0)
    if return_routing:
        return pred, routes
    return pred

pred_train = infer(dataset_traintest, BATCH_SIZE)
pred_rel = infer(dataset_reliability, BATCH_SIZE)
print(pred_train.shape, pred_rel.shape)


In [ ]:
def brec_t2(pred):
    x = pred[0::2].T
    y = pred[1::2].T
    d = x - y
    d_mean = d.mean(dim=1, keepdim=True)
    s = torch.cov(d)
    inv_s = torch.linalg.pinv(s)
    return torch.mm(torch.mm(d_mean.T, inv_s), d_mean).item()

t2_train = brec_t2(pred_train)
t2_rel = brec_t2(pred_rel)
print('T2 train/test:', t2_train)
print('T2 reliability:', t2_rel)


In [ ]:
pred, routes = infer(dataset_traintest[:2], batch_size=2, return_routing=True)
idx, scores = routes[0][0]
print('prediction shape:', pred.shape)
print('routing shape:', idx.shape)
print('selected pivots for first block:', idx[0, 0])
print('selected router scores:', scores[0, 0])


## Routing smoke-test for k = 2, 3, 4

Этот тест не использует BREC и проверяет, что один и тот же routed block работает на k-dimensional relation state.

In [ ]:
for k in (2, 3, 4):
    n = 4
    c = 16
    h = 4
    x = torch.randn(2, *([n] * k), c, device=device, dtype=torch.float64, requires_grad=True)
    block = RoutedPivotalAttentionBlock(
        embed_dim=c,
        num_heads=h,
        floyd_level=k,
        top_k=2,
        route_dim=4,
        pivot_chunk_size=2,
    ).to(device).double()
    y = block(x)
    y.mean().backward()
    print(f'k={k}: {tuple(x.shape)} -> {tuple(y.shape)}, router_grad={block.route_q.weight.grad.abs().mean().item():.6e}')
